# Local Training Notebook - DDoS Detection Models

This notebook trains MLPv2, CNN_LSTM, LSTM, and CNN1D models locally (no Federated Learning).

**Models:**
- MLPv2: Multi-Layer Perceptron
- CNN_LSTM: Hybrid CNN-LSTM
- LSTM: Long Short-Term Memory
- CNN1D: 1D Convolutional Neural Network


## 1. Install Dependencies


In [ ]:
# Install required packages
%pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn


## 2. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, Flatten, LSTM
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 3. Data Preprocessing Functions


In [ ]:
def process_col(df):
    """Clean and process columns"""
    drop_cols = ['dt', 'src', 'dst', 'switch']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    num_cols = df.select_dtypes(include=['number']).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    obj_cols = df.select_dtypes(include=['object']).columns
    if len(obj_cols) > 0:
        df[obj_cols] = df[obj_cols].fillna(df[obj_cols].mode().iloc[0])

    if 'dur' in df.columns and 'dur_nsec' in df.columns:
        df['duration_sec'] = df['dur'] + df['dur_nsec'] / 1e9
    elif 'dur' in df.columns:
        df['duration_sec'] = df['dur']
    else:
        df['duration_sec'] = 0

    num_cols = [
        'pktcount', 'bytecount', 'duration_sec', 'flows', 'packetins',
        'pktperflow', 'byteperflow', 'pktrate', 'tx_bytes', 'rx_bytes',
        'tx_kbps', 'rx_kbps', 'tot_kbps', 'port_no'
    ]
    num_cols = [c for c in num_cols if c in df.columns]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    return df


def normalization(df):
    """Normalize and engineer features"""
    df['duration_sec'] = pd.to_numeric(df.get('duration_sec', 0), errors='coerce').fillna(0)
    if 'port_no' in df.columns:
        df['port_no'] = pd.to_numeric(df['port_no'], errors='coerce')

    # Remove invalids
    for col in ['pktcount', 'bytecount', 'dur']:
        if col in df.columns:
            df = df[df[col] >= 0]

    # Encode categorical
    if 'protocol' in df.columns:
        le = LabelEncoder()
        df['protocol'] = le.fit_transform(df['protocol'])

    # Feature engineering
    df['pkt_per_sec'] = df['pktcount'] / (df['dur'] + 1e-5)
    df['byte_per_pkt'] = df['bytecount'] / (df['pktcount'] + 1e-5)
    df['rx_tx_ratio'] = (df['rx_bytes'] + 1) / (df['tx_bytes'] + 1)
    df['byte_rate'] = df['bytecount'] / (df['dur'] + 1e-5)

    # Log-scale skewed features
    for col in ['pktcount', 'bytecount', 'tx_bytes', 'rx_bytes', 'tot_kbps']:
        if col in df.columns:
            df[col] = np.log1p(df[col])

    # Outlier capping
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        df[col] = np.clip(df[col], df[col].quantile(0.01), df[col].quantile(0.99))

    df = df.dropna().reset_index(drop=True)
    return df


def split_dataset(df):
    """Split dataset into train and test"""
    X = df.drop(columns=['label'], errors='ignore')
    y = df['label']
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Upload dataset_sdn.csv to Colab or provide path
# For Colab: Use files.upload() or mount Google Drive

# Option 1: Upload file directly (uncomment below)
from google.colab import files
uploaded = files.upload()

# Option 2: If using Google Drive (uncomment below)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/dataset_sdn.csv')

# Load dataset
df = pd.read_csv('dataset_sdn.csv')
df.columns = df.columns.str.strip().str.lower()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())


In [ ]:
# Preprocess data
df = process_col(df)
df = normalization(df)

# Split dataset
X_train, X_test, y_train, y_test = split_dataset(df)

# Convert to numpy arrays
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

# Get number of features and classes
num_features = X_train.shape[1]
num_classes = len(np.unique(y_train))

# Convert labels to categorical
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# Prepare 3D data for CNN/LSTM models
X_train_3d = np.expand_dims(X_train, axis=2)
X_test_3d = np.expand_dims(X_test, axis=2)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {num_features}")
print(f"Classes: {num_classes}")
print(f"\nTraining label distribution:")
print(f"Class 0: {np.sum(y_train == 0)} ({np.sum(y_train == 0)/len(y_train)*100:.2f}%)")
print(f"Class 1: {np.sum(y_train == 1)} ({np.sum(y_train == 1)/len(y_train)*100:.2f}%)")


## 5. Model Creation Functions


In [ ]:
def create_mlpv2_model(num_features, num_classes):
    """Create MLPv2 model"""
    model = Sequential([
        keras.Input(shape=(num_features,)),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_cnn1d_model(num_features, num_classes):
    """Create CNN1D model"""
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu', input_shape=(num_features, 1)),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(128, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_lstm_model(num_features, num_classes):
    """Create LSTM model"""
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(num_features, 1)),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_cnn_lstm_model(num_features, num_classes):
    """Create CNN_LSTM hybrid model"""
    model = Sequential([
        Conv1D(64, 3, activation='relu', input_shape=(num_features, 1)),
        MaxPooling1D(2),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


## 6. Training Configuration


In [ ]:
# Training parameters
EPOCHS = 50
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

# Models to train
MODELS = {
    'MLPv2': {'create_func': create_mlpv2_model, 'use_3d': False},
    'CNN1D': {'create_func': create_cnn1d_model, 'use_3d': True},
    'LSTM': {'create_func': create_lstm_model, 'use_3d': True},
    'CNN_LSTM': {'create_func': create_cnn_lstm_model, 'use_3d': True}
}

print(f"Training configuration:")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Validation split: {VALIDATION_SPLIT}")
print(f"\nModels to train: {list(MODELS.keys())}")


## 7. Train All Models


In [ ]:
# Dictionary to store training history and models
results = {}

for model_name, model_config in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    # Create model
    model = model_config['create_func'](num_features, num_classes)
    
    # Print model summary
    print(f"\n{model_name} Architecture:")
    model.summary()
    
    # Prepare data
    if model_config['use_3d']:
        train_data = X_train_3d
        test_data = X_test_3d
    else:
        train_data = X_train
        test_data = X_test
    
    # Train model
    history = model.fit(
        train_data,
        y_train_cat,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1
    )
    
    # Evaluate on test set
    test_loss, test_accuracy = model.evaluate(test_data, y_test_cat, verbose=0)
    
    # Get predictions
    y_pred_proba = model.predict(test_data, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    # Store results
    results[model_name] = {
        'model': model,
        'history': history,
        'y_test': y_test,
        'y_pred': y_pred,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }
    
    print(f"\n{model_name} Results:")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")

print(f"\n{'='*60}")
print("All models trained successfully!")
print(f"{'='*60}")


## 8. Visualize Training History (Accuracy vs Loss)


In [ ]:
# Plot training history for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (model_name, result) in enumerate(results.items()):
    history = result['history']
    
    # Plot accuracy
    axes[idx].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[idx].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[idx].set_xlabel('Epoch', fontsize=12)
    axes[idx].set_ylabel('Accuracy', fontsize=12)
    axes[idx].set_title(f'{model_name} - Accuracy', fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Accuracy curves saved as 'accuracy_curves.png'")


In [ ]:
# Plot loss curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (model_name, result) in enumerate(results.items()):
    history = result['history']
    
    # Plot loss
    axes[idx].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[idx].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[idx].set_xlabel('Epoch', fontsize=12)
    axes[idx].set_ylabel('Loss', fontsize=12)
    axes[idx].set_title(f'{model_name} - Loss', fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Loss curves saved as 'loss_curves.png'")


## 9. Confusion Matrices


In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (model_name, result) in enumerate(results.items()):
    y_test = result['y_test']
    y_pred = result['y_pred']
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Calculate metrics
    accuracy = result['accuracy']
    precision = result['precision']
    recall = result['recall']
    f1 = result['f1']
    
    # Plot heatmap
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=[f'Class {i}' for i in range(num_classes)],
        yticklabels=[f'Class {i}' for i in range(num_classes)],
        cbar_kws={'label': 'Count'},
        ax=axes[idx],
        linewidths=0.5,
        linecolor='gray'
    )
    
    axes[idx].set_title(
        f'{model_name} Confusion Matrix\n'
        f'Accuracy: {accuracy*100:.2f}% | Precision: {precision*100:.2f}% | '
        f'Recall: {recall*100:.2f}% | F1: {f1*100:.2f}%',
        fontsize=12,
        fontweight='bold'
    )
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    axes[idx].set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrices saved as 'confusion_matrices.png'")


## 10. Metrics Comparison


In [ ]:
# Create comparison table
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{result['accuracy']*100:.2f}%",
        'Precision': f"{result['precision']*100:.2f}%",
        'Recall': f"{result['recall']*100:.2f}%",
        'F1 Score': f"{result['f1']*100:.2f}%",
        'Test Loss': f"{result['test_loss']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*60)
print("Metrics Comparison")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)


In [ ]:
# Visualize metrics comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    model_names = list(results.keys())
    metric_values = [results[name][metric] * 100 for name in model_names]
    
    bars = axes[idx].bar(model_names, metric_values, color=['#4caf50', '#2196f3', '#ff9800', '#f44336'])
    axes[idx].set_ylabel(f'{metric.capitalize()} (%)', fontsize=12)
    axes[idx].set_title(f'{metric.capitalize()} Comparison', fontsize=14, fontweight='bold')
    axes[idx].set_ylim([0, 100])
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{height:.2f}%',
                      ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Metrics comparison saved as 'metrics_comparison.png'")


## 11. Detailed Classification Reports


In [ ]:
# Print detailed classification reports
for model_name, result in results.items():
    print(f"\n{'='*60}")
    print(f"{model_name} - Detailed Classification Report")
    print(f"{'='*60}")
    print(classification_report(
        result['y_test'],
        result['y_pred'],
        target_names=[f'Class {i}' for i in range(num_classes)]
    ))
    print(f"{'='*60}")


## 12. Save Models (Optional)


In [ ]:
# Save all trained models
for model_name, result in results.items():
    filename = f"{model_name}_local.h5"
    result['model'].save(filename)
    print(f"Saved {model_name} model as {filename}")

print("\nAll models saved successfully!")
